In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interactive, FloatSlider, IntSlider, ToggleButtons
from IPython.display import display, clear_output


def _bode_fig():
    # Returns (fig, ax_magnitude, ax_phase) with shared formatting
    fig, (ax_m, ax_p) = plt.subplots(2, 1, figsize=(10, 6))
    for ax in (ax_m, ax_p):
        ax.grid(True, which='both', alpha=0.35)
    ax_m.set_ylabel('Magnitude (dB)')
    ax_p.set_ylabel('Phase (rad)')
    ax_p.set_xlabel('Frequency (rad/s)')
    ax_p.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax_p.set_yticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
    return fig, ax_m, ax_p


def _margins(freq, mag, phase):
    # Numerically estimate gain margin [dB] and phase margin [deg]
    # from Bode data: freq [rad/s], mag [dB], phase [deg]
    shifted = phase + 180
    idx_g = np.where(np.diff(np.sign(shifted)))[0]
    if len(idx_g):
        i = idx_g[0]
        a = -shifted[i] / (shifted[i+1] - shifted[i])
        wg = np.exp(np.log(freq[i]) + a * np.log(freq[i+1] / freq[i]))
        GM = float(-np.interp(np.log10(wg), np.log10(freq), mag))
    else:
        wg, GM = np.nan, np.inf
    idx_c = np.where(np.diff(np.sign(mag)))[0]
    if len(idx_c):
        i = idx_c[0]
        a = -mag[i] / (mag[i+1] - mag[i])
        wc = freq[i] + a * (freq[i+1] - freq[i])
        PM = float(np.interp(wc, freq, phase)) + 180
    else:
        wc, PM = np.nan, np.inf
    return GM, PM, wg, wc

# Bode Diagram

The **Bode diagram** visualises the **frequency response** of a linear time-invariant (LTI) system $G(s)$, obtained by evaluating $G$ on the imaginary axis $s = j\omega$:

$$G(j\omega) = |G(j\omega)|\,e^{j\phi(\omega)}$$

| Subplot | Definition | Unit |
|---|---|---|
| **Magnitude** | $\|G(j\omega)\|_{\text{dB}} = 20\log_{10}\|G(j\omega)\|$ | dB |
| **Phase** | $\phi(\omega) = \angle G(j\omega)$ | rad |

Frequency is plotted on a **logarithmic axis** (rad/s). For a factored transfer function $G = K\cdot G_1 \cdot G_2 \cdots$:

$$\|G\|_{\text{dB}} = 20\log_{10}K + \|G_1\|_{\text{dB}} + \|G_2\|_{\text{dB}} + \cdots\qquad \phi = \phi_1 + \phi_2 + \cdots$$

Each elementary factor contributes an **additive, independent** piece, enabling fast piecewise-linear **asymptotic approximations** that can be composed graphically.

## Pure Gain — $G(s) = K$

A static gain introduces no dynamics.  The amplitude ratio is constant at all frequencies and the phase is identically zero:

$$|G(j\omega)|_{\text{dB}} = 20\log_{10}K \quad (\text{independent of }\omega)\qquad \phi(\omega) = 0°$$

On the Bode diagram this appears as a **horizontal line**. $K > 1$ lifts it above 0 dB; $K < 1$ places it below 0 dB. Varying $K$ is a pure vertical shift — it never changes slope or phase.

In [ ]:
def _plot_gain(K):
    clear_output(wait=True)
    freq = np.logspace(-2, 2, 1000)
    _, mag, phase = signal.bode(signal.TransferFunction([K], [1]), freq)
    K_dB = 20 * np.log10(K)

    fig, ax_m, ax_p = _bode_fig()

    ax_m.semilogx(freq, mag, 'C0', lw=2.5)
    ax_m.axhline(K_dB, color='C3', ls='--', lw=1.5,
                 label=f'K = {K:.2f}  →  {K_dB:+.1f} dB')
    ax_m.axhline(0, color='k', lw=0.7, ls=':')
    ax_m.set_ylim(-40, 40)
    ax_m.set_title(f'G(s) = K = {K:.2f}')
    ax_m.legend()

    ax_p.semilogx(freq, phase * np.pi / 180, 'C1', lw=2.5)
    ax_p.set_ylim(-np.pi / 2, np.pi / 2)

    plt.tight_layout()
    plt.show()


display(interactive(_plot_gain,
    K=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                  description='K:', style={'description_width': 'initial'})))

interactive(children=(FloatSlider(value=1.0, description='K:', max=10.0, min=0.1, style=SliderStyle(descriptio…

## Poles and Zeros at the Origin — $G(s) = K s^n$

Each **zero at the origin** adds $+20$ dB/dec to the magnitude slope and $+90°$ to the (constant) phase. Each **pole** reverses the signs:

| Transfer function | Magnitude slope | Constant phase |
|---|---|---|
| $K s^n$, $n > 0$ (zeros) | $+20n$ dB/dec | $+n \cdot 90°$ |
| $K$ | 0 dB/dec | 0° |
| $K / s^n$, $n > 0$ (poles) | $-20n$ dB/dec | $-n \cdot 90°$ |

At $\omega = 1$ rad/s the magnitude always equals $20\log_{10}K$ dB, regardless of $n$. The asymptote is exact for all $\omega$:

$$|G(j\omega)|_{\text{dB}} = 20\log_{10}K + 20n\log_{10}\omega$$

Use the slider below with **positive $n$** for zeros and **negative $n$** for poles.

In [ ]:
def _plot_origin(K, n):
    clear_output(wait=True)
    if n >= 0:
        num_c = [K] + [0] * n
        den_c = [1]
        tf_label = f'G(s) = K·s^{n}' if n > 0 else 'G(s) = K'
    else:
        num_c = [K]
        den_c = [1] + [0] * (-n)
        tf_label = f'G(s) = K / s^{-n}'

    freq = np.logspace(-2, 2, 1000)
    _, mag, phase = signal.bode(signal.TransferFunction(num_c, den_c), freq)

    K_dB = 20 * np.log10(K)
    mag_asym = K_dB + 20 * n * np.log10(freq)
    slope_str = f'{20 * n:+.0f} dB/dec' if n != 0 else '0 dB/dec (flat)'

    fig, ax_m, ax_p = _bode_fig()

    ax_m.semilogx(freq, mag, 'C0', lw=2.5, label='Exact')
    ax_m.semilogx(freq, mag_asym, 'C2--', lw=1.5,
                  label=f'Asymptote: {slope_str}')
    ax_m.axhline(K_dB, color='C3', ls=':', lw=1.2,
                 label=f'20⋅log₁₀(K) = {K_dB:.1f} dB at ω=1')
    ax_m.set_ylim(-80, 80)
    ax_m.set_title(f'{tf_label},   K = {K:.1f},   n = {n}')
    ax_m.legend(fontsize=9)

    phase_const = n * np.pi / 2
    ax_p.semilogx(freq, phase * np.pi / 180, 'C1', lw=2.5)
    ax_p.axhline(phase_const, color='C2', ls='--', lw=1.5,
                 label=f'Phase = {n}·90° = {phase_const/np.pi:.1f}π rad')
    ax_p.set_ylim(-2 * np.pi, 2 * np.pi)
    ax_p.set_yticks([-3*np.pi/2, -np.pi, -np.pi/2, 0, np.pi/2, np.pi, 3*np.pi/2])
    ax_p.set_yticklabels(['-3π/2', '-π', '-π/2', '0', 'π/2', 'π', '3π/2'])
    ax_p.legend(fontsize=9)

    plt.tight_layout()
    plt.show()


display(interactive(_plot_origin,
    K=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                  description='K:', style={'description_width': 'initial'}),
    n=IntSlider(value=1, min=-3, max=3, step=1,
                description='n  (neg = poles):', style={'description_width': 'initial'})))

interactive(children=(FloatSlider(value=1.0, description='K:', max=10.0, min=0.1, style=SliderStyle(descriptio…

## First-Order Real Pole / Zero — $G(s) = K\,/\,(1+s\tau)$ or $K(1+s\tau)$

The **break frequency** $\omega_b = 1/\tau$ separates two asymptotic regions.

**Real pole** $G(s) = K/(1+s\tau)$:

$$|G(j\omega)|_{\text{dB}} \approx \begin{cases} 20\log_{10}K & \omega \ll \omega_b \\ 20\log_{10}K - 20\log_{10}(\omega/\omega_b) & \omega \gg \omega_b \end{cases}\qquad \phi(\omega_b) = -45°$$

**Real zero** $G(s) = K(1+s\tau)$: all slopes and phase shifts are **reversed** ($+20$ dB/dec, phase $0°\to+90°$).

The asymptote approximation has a maximum error of $\approx 3$ dB at $\omega_b$. The phase transition spans roughly one decade on each side of $\omega_b$.

In [4]:
def _plot_first_order(K, tau, mode):
    clear_output(wait=True)
    wb = 1.0 / tau
    K_dB = 20 * np.log10(K)
    freq = np.logspace(-2, 2, 2000)
    sign = -1 if mode == 'Pole' else +1

    if mode == 'Pole':
        sys_tf = signal.TransferFunction([K], [tau, 1])
        tf_label = f'G(s) = {K:.1f} / (1 + {tau:.1f}s)'
    else:
        sys_tf = signal.TransferFunction([K * tau, K], [1])
        tf_label = f'G(s) = {K:.1f} · (1 + {tau:.1f}s)'

    _, mag, phase = signal.bode(sys_tf, freq)

    # Piecewise-linear magnitude asymptote
    mag_asym = np.where(
        freq <= wb,
        K_dB,
        K_dB + sign * 20 * np.log10(freq / wb)
    )

    fig, ax_m, ax_p = _bode_fig()

    ax_m.semilogx(freq, mag, 'C0', lw=2.5, label='Exact')
    ax_m.semilogx(freq, mag_asym, 'C2--', lw=1.5,
                  label=f'Asymptote ({sign*20:+.0f} dB/dec above ωb)')
    ax_m.axvline(wb, color='C3', ls=':', lw=1.5,
                 label=f'ωb = 1/τ = {wb:.2f} rad/s')
    ax_m.set_ylim(-40, 40)
    ax_m.set_title(tf_label)
    ax_m.legend(fontsize=9)

    ax_p.semilogx(freq, phase * np.pi / 180, 'C1', lw=2.5)
    ax_p.axvline(wb, color='C3', ls=':', lw=1.5,
                 label=f'Phase = {sign*45:.0f}° at ωb')
    ax_p.set_ylim(-np.pi * 0.65, np.pi * 0.65)
    ax_p.legend(fontsize=9)

    plt.tight_layout()
    plt.show()


display(interactive(_plot_first_order,
    K=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                  description='K:', style={'description_width': 'initial'}),
    tau=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                    description='τ (time constant, s):', style={'description_width': 'initial'}),
    mode=ToggleButtons(options=['Pole', 'Zero'], value='Pole',
                       description='Type:', style={'description_width': 'initial'})))

interactive(children=(FloatSlider(value=1.0, description='K:', max=10.0, min=0.1, style=SliderStyle(descriptio…

## Second-Order Complex Pole / Zero Pair

Standard form with **natural frequency** $\omega_n$ and **damping ratio** $\zeta$:

$$
G(s) = \frac{K\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}
\qquad \text{(poles)}
$$

$$
G(s) = \frac{K}{\omega_n^2}\left(s^2 + 2\zeta\omega_n s + \omega_n^2\right)
\qquad \text{(zeros)}
$$

| $\zeta$ range | Poles | Magnitude behavior near $\omega_n$ |
|---|---|---|
| $\zeta > 1$ | Two real poles | Two successive $-20$ dB/dec breaks |
| $\zeta = 1$ | Repeated real pole | Smooth transition to $-40$ dB/dec |
| $0 < \zeta < \frac{1}{\sqrt{2}}$ | Complex conjugate poles | **Resonance peak** visible |

For the underdamped case, the resonance peak occurs at:

$$
\omega_r = \omega_n \sqrt{1 - 2\zeta^2}
$$

$$
M_r = \frac{K}{2\zeta\sqrt{1-\zeta^2}}
\qquad [\text{linear scale}]
$$

The phase transitions from **$0^\circ$ to $-180^\circ$** (poles) or **$0^\circ$ to $+180^\circ$** (zeros), reaching **$\pm 90^\circ$** exactly at $\omega_n$.

In [5]:
def _plot_second_order(K, zeta, wn, mode):
    clear_output(wait=True)
    freq = np.logspace(-1, 2, 2000)

    if mode == 'Poles':
        sys_tf = signal.TransferFunction([K * wn**2], [1, 2*zeta*wn, wn**2])
        tf_label = f'G(s) = Kωn² / (s² + 2ζωns + ωn²)'
    else:
        sys_tf = signal.TransferFunction(
            [K/wn**2, 2*K*zeta/wn, K], [1])
        tf_label = f'G(s) = (K/ωn²)(s² + 2ζωns + ωn²)'

    _, mag, phase = signal.bode(sys_tf, freq)

    fig, ax_m, ax_p = _bode_fig()

    ax_m.semilogx(freq, mag, 'C0', lw=2.5)
    ax_m.axvline(wn, color='C3', ls='--', lw=1.5,
                 label=f'ωn = {wn:.1f} rad/s')

    if mode == 'Poles' and zeta < 1.0 / np.sqrt(2):
        wr = wn * np.sqrt(max(0.0, 1 - 2 * zeta**2))
        Mr_dB = float(np.interp(wr, freq, mag))
        ax_m.plot(wr, Mr_dB, 'C1o', ms=9, zorder=5,
                  label=f'ωr = {wr:.2f} rad/s,  Mr = {Mr_dB:.1f} dB')
        ax_p.axvline(wr, color='C1', ls=':', lw=1.2, alpha=0.7)

    ax_m.set_ylim(-60, 40)
    title_line2 = f'K={K:.1f},  ζ={zeta:.2f},  ωn={wn:.1f} rad/s'
    ax_m.set_title(tf_label + '\n' + title_line2)
    ax_m.legend(fontsize=9)

    ax_p.semilogx(freq, phase * np.pi / 180, 'C1', lw=2.5)
    ax_p.axvline(wn, color='C3', ls='--', lw=1.5, alpha=0.6)
    if mode == 'Poles':
        ax_p.set_ylim(-np.pi * 1.05, 0.15)
    else:
        ax_p.set_ylim(-0.15, np.pi * 1.05)

    plt.tight_layout()
    plt.show()


display(interactive(_plot_second_order,
    K=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                  description='K:', style={'description_width': 'initial'}),
    zeta=FloatSlider(value=0.3, min=0.01, max=2.0, step=0.01,
                     description='ζ (damping):', style={'description_width': 'initial'}),
    wn=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                   description='ωn (nat. freq.):', style={'description_width': 'initial'}),
    mode=ToggleButtons(options=['Poles', 'Zeros'], value='Poles',
                       description='Type:', style={'description_width': 'initial'})))

interactive(children=(FloatSlider(value=1.0, description='K:', max=10.0, min=0.1, style=SliderStyle(descriptio…

## Gain and Phase Margins

For a unity-feedback loop with open-loop $G(s)$, **stability margins** quantify how far the Nyquist curve of $G(j\omega)$ is from the critical point $-1$:

$$\omega_c : |G(j\omega_c)| = 0\text{ dB} \quad \Rightarrow \quad\boxed{\text{Phase Margin} = 180° + \angle G(j\omega_c)}$$

$$\omega_{\phi} : \angle G(j\omega_{\phi}) = -180° \quad \Rightarrow \quad\boxed{\text{Gain Margin} = -|G(j\omega_{\phi})|_{\text{dB}}}$$

Both margins must be **positive** for closed-loop stability (minimum-phase systems). A practical design target: **PM > 45°**, **GM > 6 dB**.

The plant below is $G(s) = K\,/\,[s(\tau_1 s + 1)(\tau_2 s + 1)]$ — a Type-1 third-order system typical in servo and process control. Increase $K$ to push the system towards instability and observe how both margins shrink.

In [6]:
def _plot_margins(K, tau1, tau2):
    clear_output(wait=True)
    # G(s) = K / [ s (tau1*s+1)(tau2*s+1) ]
    sys_tf = signal.TransferFunction(
        [K],
        [tau1*tau2, tau1+tau2, 1, 0]
    )
    freq = np.logspace(-2, 2, 4000)
    _, mag, phase = signal.bode(sys_tf, freq)

    GM, PM, wg, wc = _margins(freq, mag, phase)

    stable = (GM > 0) and (PM > 0)
    s_color = 'tab:green' if stable else 'tab:red'
    s_str   = 'STABLE' if stable else 'UNSTABLE'

    fig, ax_m, ax_p = _bode_fig()

    ax_m.semilogx(freq, mag, 'C0', lw=2)
    ax_m.axhline(0, color='k', lw=0.7, ls=':')
    if not np.isnan(wg):
        ax_m.axvline(wg, color='gray', ls='--', lw=1, alpha=0.6)
        ax_m.plot([wg, wg], [0, -GM], color='C3', lw=2.5,
                  label=f'GM = {GM:.1f} dB   (ωφ = {wg:.2f} rad/s)')
    ax_m.set_ylim(-60, 40)
    ax_m.set_title(
        f'G(s) = K / [s(τ1s+1)(τ2s+1)]   '
        f'K={K:.1f}  τ1={tau1:.1f}  τ2={tau2:.1f}   [{s_str}]',
        color=s_color, fontsize=10)
    ax_m.legend(fontsize=9, loc='lower left')

    ax_p.semilogx(freq, phase * np.pi / 180, 'C1', lw=2)
    ax_p.axhline(-np.pi, color='k', lw=0.7, ls=':')
    if not np.isnan(wc):
        ph_wc = float(np.interp(wc, freq, phase)) * np.pi / 180
        ax_p.axvline(wc, color='gray', ls='--', lw=1, alpha=0.6)
        ax_p.plot([wc, wc], [-np.pi, ph_wc], color='C2', lw=2.5,
                  label=f'PM = {PM:.1f}°   (ωc = {wc:.2f} rad/s)')
    ax_p.set_ylim(-3 * np.pi / 2, 0.2)
    ax_p.set_yticks([-3*np.pi/2, -np.pi, -np.pi/2, 0])
    ax_p.set_yticklabels(['-3π/2', '-π', '-π/2', '0'])
    ax_p.legend(fontsize=9, loc='lower left')

    plt.tight_layout()
    plt.show()


display(interactive(_plot_margins,
    K=FloatSlider(value=1.0, min=0.1, max=20.0, step=0.1,
                  description='K:', style={'description_width': 'initial'}),
    tau1=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                     description='τ1:', style={'description_width': 'initial'}),
    tau2=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                     description='τ2:', style={'description_width': 'initial'})))

interactive(children=(FloatSlider(value=1.0, description='K:', max=20.0, min=0.1, style=SliderStyle(descriptio…